# Energy Commodities & Geopolitical Risk: An Event Study

**Author:** Tanmay Agrawal · BSc Accounting & Finance, Queen Mary University of London

**Question:** When a geopolitical shock hits the Middle East or another energy chokepoint, how much do oil, gas, freight, tanker and coal-equity prices move, and is the move statistically significant?

**Method:** A classic event study (MacKinlay, 1997). For each event, it estimates each asset's "normal" return over a clean pre-event window, measures the *abnormal* return around the event, cumulates it (CAR), and tests significance across events with a cross-sectional t-test, a sign test, and bootstrap confidence intervals.

**Assets:** Brent crude · WTI crude · Henry Hub natural gas · dry-bulk freight (BDRY) · crude tankers (Frontline) · coal equity (Peabody) · energy sector (XLE) · gold (safe-haven control) · S&P 500 (market).

> Runs end-to-end in Google Colab: **Runtime → Run all** (≈1–2 minutes). No API keys needed.

## 1 · Setup and installs

In [ ]:
# Colab ships with pandas, numpy, scipy and matplotlib; yfinance is the only add-on.
!pip install -q -U yfinance curl_cffi

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import os
import warnings
from dataclasses import dataclass, field
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import TwoSlopeNorm
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# ── Reproducibility & output folders ───────────────────────────────────────
SEED = 42
rng = np.random.default_rng(SEED)
OUT_DIR = "outputs"
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)
print("Environment ready ✔")

## 2 · Configuration
Every research decision lives here, so the study can be re-run under different assumptions without touching the logic.

| Parameter | Default | Why |
|---|---|---|
| Estimation window | 120 trading days, ending 15 days before the event (5-day pre-window + 10-day gap) | Long enough for a stable mean/beta; the gap keeps pre-event news leakage out |
| Event window | −5 to +10 trading days | Captures anticipation and the post-shock drift |
| Model | `market` (beta to S&P 500) or `mean` (constant mean) | Commodities have low equity beta, so both are reported |

In [ ]:
@dataclass(frozen=True)
class Config:
    start: str = "2017-01-01"
    end: str | None = None                  # None = today
    est_len: int = 120                      # estimation window length (trading days)
    gap: int = 10                           # buffer between estimation and event windows
    pre: int = 5                            # days before event in event window
    post: int = 10                          # days after event in event window
    model: str = "market"                   # "market" or "mean"
    market: str = "SPY"
    n_boot: int = 5_000                     # bootstrap resamples for CIs
    alpha: float = 0.05
    assets: dict = field(default_factory=lambda: {
        "BZ=F": "Brent crude",
        "CL=F": "WTI crude",
        "NG=F": "US natural gas",
        "BDRY": "Dry-bulk freight",
        "FRO":  "Crude tankers (Frontline)",
        "BTU":  "Coal equity (Peabody)",
        "XLE":  "Energy sector",
        "GC=F": "Gold (safe haven)",
    })

CFG = Config()
CFG

## 3 · Event catalogue
Hand-curated geopolitical shocks to energy supply routes. Dates are the day the news broke; weekend events map automatically to the next trading day. **Check each date against a primary news source before publishing**, and add your own events by editing the CSV text.

Categories: `chokepoint` (Hormuz / Red Sea shipping), `conflict` (state-level military action), `supply` (production or OPEC+ policy shock).

In [ ]:
EVENTS_CSV = """date,event,category
2019-05-12,Fujairah tanker sabotage (UAE),chokepoint
2019-06-13,Gulf of Oman tanker attacks,chokepoint
2019-06-20,Iran shoots down US drone near Hormuz,chokepoint
2019-07-19,Iran seizes tanker Stena Impero,chokepoint
2019-09-14,Abqaiq-Khurais attack on Saudi oil facilities,supply
2020-01-03,US strike kills Qasem Soleimani,conflict
2020-03-06,OPEC+ talks collapse (price war),supply
2022-02-24,Russia invades Ukraine,conflict
2023-04-02,Surprise OPEC+ production cut,supply
2023-10-07,Hamas attack on Israel,conflict
2023-11-19,Houthis seize ship Galaxy Leader in Red Sea,chokepoint
2024-01-12,US-UK strikes on Houthis in Yemen,chokepoint
2024-04-13,Iran drone and missile attack on Israel,conflict
2024-10-01,Iran ballistic missile attack on Israel,conflict
2025-06-13,Israel strikes Iran,conflict
2025-06-22,US strikes Iranian nuclear sites,conflict
"""

def load_events(csv_text: str) -> pd.DataFrame:
    """Parse and validate the event catalogue."""
    ev = pd.read_csv(StringIO(csv_text), parse_dates=["date"])
    required = {"date", "event", "category"}
    missing = required - set(ev.columns)
    if missing:
        raise ValueError(f"Event CSV missing columns: {missing}")
    if ev["date"].isna().any():
        raise ValueError("Unparseable dates in event catalogue")
    if ev.duplicated("date").any():
        raise ValueError("Duplicate event dates: merge them or offset by a day")
    return ev.sort_values("date").reset_index(drop=True)

EVENTS = load_events(EVENTS_CSV)
EVENTS

## 4 · Data collection pipeline
Downloads daily adjusted closes from Yahoo Finance in **one batched request**, caches them to `data/prices.parquet` (or CSV if parquet support is missing), and retries with backoff if the request fails, so re-running the notebook doesn't hit the API again.

In [ ]:
import time
import yfinance as yf

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

def download_prices(tickers: list[str], start: str, end: str | None,
                    retries: int = 3, cache: str = os.path.join(DATA_DIR, "prices")) -> pd.DataFrame:
    """Fetch adjusted close prices with caching and exponential-backoff retries.

    Returns a DataFrame indexed by date with one column per ticker.
    Raises RuntimeError if no data could be retrieved at all.
    """
    for ext, reader in ((".parquet", pd.read_parquet), (".csv", lambda p: pd.read_csv(p, index_col=0, parse_dates=True))):
        path = cache + ext
        if os.path.exists(path):
            cached = reader(path)
            if set(tickers).issubset(cached.columns):
                print(f"Loaded cached prices from {path}")
                return cached[tickers]

    last_err = None
    for attempt in range(1, retries + 1):
        try:
            raw = yf.download(tickers, start=start, end=end, auto_adjust=True,
                              progress=False, group_by="column", threads=True)
            if raw.empty:
                raise RuntimeError("Empty response from Yahoo Finance")
            px = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw[["Close"]].rename(columns={"Close": tickers[0]})
            px = px.reindex(columns=tickers)
            break
        except Exception as e:                       # network / rate-limit / parsing
            last_err = e
            wait = 2 ** attempt
            print(f"Attempt {attempt}/{retries} failed ({e}); retrying in {wait}s")
            time.sleep(wait)
    else:
        raise RuntimeError(f"Could not download prices: {last_err}")

    empty = [t for t in tickers if px[t].dropna().empty]
    if empty:
        print(f"⚠ No data for {empty}: they will be dropped")
        px = px.drop(columns=empty)
    try:
        px.to_parquet(cache + ".parquet")
    except Exception:
        px.to_csv(cache + ".csv")
    return px

TICKERS = list(CFG.assets) + [CFG.market]
prices = download_prices(TICKERS, CFG.start, CFG.end)
print(prices.shape, prices.index.min().date(), "→", prices.index.max().date())
prices.tail()

## 5 · Data cleaning and feature engineering
* **Calendar alignment:** futures and equities trade on slightly different holidays, so everything is aligned to the S&P 500 trading calendar, and short gaps (≤3 days) are forward-filled.
* **Log returns:** additive across days, so CARs are simple sums.
* **Negative prices:** WTI settled at −$37 on 20 Apr 2020, which makes a log return undefined. Non-positive prices are masked, and that day's return becomes NaN rather than ±∞.
* **Outlier report:** daily moves beyond 8 standard deviations are flagged, not deleted. Deleting them would hide exactly the shocks being studied.
* **Event-day mapping:** each event date maps to the first trading day on or after it.

In [ ]:
def clean_prices(px: pd.DataFrame, market: str, max_ffill: int = 3) -> pd.DataFrame:
    """Align to market calendar, mask non-positive prices, forward-fill short gaps."""
    if market not in px:
        raise KeyError(f"Market ticker {market} missing from prices")
    cal = px[market].dropna().index
    out = px.reindex(cal)
    out = out.where(out > 0)                 # handles WTI's negative print
    return out.ffill(limit=max_ffill)

def log_returns(px: pd.DataFrame) -> pd.DataFrame:
    r = np.log(px).diff()
    return r.replace([np.inf, -np.inf], np.nan).iloc[1:]

def flag_outliers(r: pd.DataFrame, z: float = 8.0) -> pd.DataFrame:
    zs = (r - r.mean()) / r.std()
    hits = zs.abs().stack()
    hits = hits[hits > z]
    return hits.rename("abs_z").reset_index().rename(columns={"level_0": "date", "level_1": "ticker"})

def map_event_days(events: pd.DataFrame, index: pd.DatetimeIndex) -> pd.DataFrame:
    """Snap each event to the first trading day on/after its date; drop out-of-sample events."""
    ev = events.copy()
    pos = index.searchsorted(ev["date"])
    ev["t0_pos"] = pos
    ev = ev[ev["t0_pos"] < len(index)].copy()
    ev["t0"] = index[ev["t0_pos"]]
    return ev

px_clean = clean_prices(prices, CFG.market)
rets = log_returns(px_clean)
EV = map_event_days(EVENTS, rets.index)

print(f"Returns: {rets.shape[0]:,} days × {rets.shape[1]} assets")
print(f"Events in sample: {len(EV)}/{len(EVENTS)}")
print("\nMissing-data share per asset:\n", rets.isna().mean().round(4).to_string())
print("\nExtreme moves (|z| > 8), kept, not deleted:")
flag_outliers(rets).round({"abs_z": 2})

## 6 · Core model: abnormal returns and CARs
For asset *i* and event *e*, with event-time τ ∈ [−5, +10]:

$$AR_{i,\tau} = R_{i,\tau} - (\hat\alpha_i + \hat\beta_i R_{m,\tau}) \qquad CAR_i(\tau_1,\tau_2)=\sum_{\tau=\tau_1}^{\tau_2} AR_{i,\tau}$$

α and β are estimated by OLS on the estimation window only. Under the constant-mean model, β = 0. The estimation-window residual σ is also stored, to standardise ARs (the Patell / BMP approach) and to scale the test statistics.

In [ ]:
@dataclass
class EventResult:
    ar: pd.DataFrame        # MultiIndex (event, tau) × asset: abnormal returns
    sigma: pd.DataFrame     # event × asset: estimation-window residual std
    meta: pd.DataFrame      # event metadata (dates, category)

def estimate_normal_model(y: pd.Series, x: pd.Series, model: str) -> tuple[float, float, float]:
    """Return (alpha, beta, residual_std) from the estimation window."""
    df = pd.concat([y, x], axis=1).dropna()
    if len(df) < 30:
        return np.nan, np.nan, np.nan
    if model == "mean":
        a, b = df.iloc[:, 0].mean(), 0.0
    elif model == "market":
        b, a = np.polyfit(df.iloc[:, 1], df.iloc[:, 0], 1)
    else:
        raise ValueError("model must be 'market' or 'mean'")
    resid = df.iloc[:, 0] - (a + b * df.iloc[:, 1])
    return a, b, resid.std(ddof=2 if model == "market" else 1)

def run_event_study(rets: pd.DataFrame, ev: pd.DataFrame, cfg: Config) -> EventResult:
    """Vectorised over assets, looped over events (n_events is small)."""
    assets = [c for c in rets.columns if c != cfg.market]
    taus = np.arange(-cfg.pre, cfg.post + 1)
    R, M = rets[assets].to_numpy(), rets[cfg.market].to_numpy()
    ar_blocks, sig_rows, keep = [], [], []

    for _, e in ev.iterrows():
        p0 = int(e["t0_pos"]) - 1          # returns index is shifted by one vs prices
        est_end = p0 - cfg.pre - cfg.gap
        est_start = est_end - cfg.est_len
        if est_start < 0 or p0 + cfg.post >= len(R):
            print(f"Skipping {e['event']}: not enough history/future data")
            continue

        est_slice, win_slice = slice(est_start, est_end), slice(p0 - cfg.pre, p0 + cfg.post + 1)
        ar_mat, sig_vec = np.full((len(taus), len(assets)), np.nan), np.full(len(assets), np.nan)
        for j, a in enumerate(assets):
            alpha, beta, sig = estimate_normal_model(
                pd.Series(R[est_slice, j]), pd.Series(M[est_slice]), cfg.model)
            ar_mat[:, j] = R[win_slice, j] - (alpha + beta * M[win_slice])
            sig_vec[j] = sig

        ar_blocks.append(pd.DataFrame(ar_mat, columns=assets,
                         index=pd.MultiIndex.from_product([[e["event"]], taus], names=["event", "tau"])))
        sig_rows.append(pd.Series(sig_vec, index=assets, name=e["event"]))
        keep.append(e)

    if not ar_blocks:
        raise RuntimeError("No events had sufficient data")
    meta = pd.DataFrame(keep).set_index("event")[["date", "t0", "category"]]
    return EventResult(pd.concat(ar_blocks), pd.DataFrame(sig_rows), meta)

res = run_event_study(rets, EV, CFG)
print(f"Abnormal returns computed for {res.meta.shape[0]} events × {res.ar.shape[1]} assets")
res.ar.xs(0, level="tau").round(4).head()

## 7 · Statistical testing
Three tests on each asset's CAR across events, because no single test is robust to everything:

1. **Cross-sectional t-test.** Uses the dispersion of CARs across events, so it's robust to event-induced volatility (the variance spikes when shocks hit).
2. **Standardised (BMP) test.** Boehmer, Musumeci & Poulsen (1991). Each CAR is scaled by its own estimation σ·√L, so noisy assets don't dominate.
3. **Sign test.** Non-parametric: are more than half the CARs positive? It's robust to a single huge event (for example, Russia–Ukraine).

Plus a **bootstrap 95% CI** for the mean CAR (5,000 resamples).

In [ ]:
def car(ar: pd.DataFrame, t1: int, t2: int) -> pd.DataFrame:
    """Cumulative abnormal return per event over [t1, t2]; NaN if any day is missing."""
    tau = ar.index.get_level_values("tau")
    win = ar[(tau >= t1) & (tau <= t2)]
    return win.groupby(level="event", sort=False).sum(min_count=t2 - t1 + 1)

def bootstrap_ci(x: np.ndarray, n: int, alpha: float, rng) -> tuple[float, float]:
    x = x[~np.isnan(x)]
    if len(x) < 3:
        return np.nan, np.nan
    means = rng.choice(x, size=(n, len(x)), replace=True).mean(axis=1)
    return tuple(np.quantile(means, [alpha / 2, 1 - alpha / 2]))

def test_car(res: EventResult, t1: int, t2: int, cfg: Config, rng) -> pd.DataFrame:
    C = car(res.ar, t1, t2)
    L = t2 - t1 + 1
    scar = C / (res.sigma.loc[C.index] * np.sqrt(L))      # standardised CARs
    rows = []
    for a in C.columns:
        x, z = C[a].dropna().to_numpy(), scar[a].dropna().to_numpy()
        n = len(x)
        if n < 3:
            continue
        t_cs, p_cs = stats.ttest_1samp(x, 0.0)
        t_bmp = z.mean() / (z.std(ddof=1) / np.sqrt(len(z)))
        p_bmp = 2 * stats.t.sf(abs(t_bmp), len(z) - 1)
        p_sign = stats.binomtest(int((x > 0).sum()), n, 0.5).pvalue
        lo, hi = bootstrap_ci(x, cfg.n_boot, cfg.alpha, rng)
        rows.append(dict(asset=a, name=cfg.assets.get(a, a), n=n,
                         mean_CAR=x.mean(), median_CAR=np.median(x),
                         ci_low=lo, ci_high=hi, pct_positive=(x > 0).mean(),
                         t_cs=t_cs, p_cs=p_cs, t_bmp=t_bmp, p_bmp=p_bmp, p_sign=p_sign))
    out = pd.DataFrame(rows).set_index("asset")
    out["significant"] = (out["p_cs"] < cfg.alpha) | (out["p_bmp"] < cfg.alpha)
    return out.sort_values("mean_CAR", ascending=False)

WINDOWS = {"[-1,+1]": (-1, 1), "[0,+5]": (0, 5), "[-5,+10]": (-CFG.pre, CFG.post)}
summary = {k: test_car(res, *w, CFG, rng) for k, w in WINDOWS.items()}

key = "[0,+5]"
print(f"Main window {key}, model = {CFG.model}")
summary[key][["name", "n", "mean_CAR", "ci_low", "ci_high", "pct_positive", "p_cs", "p_bmp", "p_sign", "significant"]]

## 8 · Robustness: by event category and by model

In [ ]:
def car_by_category(res: EventResult, t1: int, t2: int) -> pd.DataFrame:
    C = car(res.ar, t1, t2).join(res.meta["category"])
    return C.groupby("category").mean().T

cat_table = car_by_category(res, 0, 5)
print("Mean CAR[0,+5] by event category:")
display(cat_table.style.format("{:+.2%}").background_gradient(cmap="RdBu_r", axis=None, vmin=-0.1, vmax=0.1))

# Re-run with the constant-mean model to check results don't hinge on the market model
alt_cfg = Config(model="mean" if CFG.model == "market" else "market")
alt = test_car(run_event_study(rets, EV, alt_cfg), 0, 5, alt_cfg, rng)
robust = summary[key][["mean_CAR", "p_cs"]].join(alt[["mean_CAR", "p_cs"]], rsuffix=f"_{alt_cfg.model}")
print(f"\nModel robustness, CAR[0,+5]: {CFG.model} vs {alt_cfg.model}")
robust

## 9 · Volatility regime: does risk rise after the shock?
Compares realised volatility in the 20 trading days before vs after each event (annualised), with a paired Wilcoxon signed-rank test.

In [ ]:
def vol_shift(rets: pd.DataFrame, ev: pd.DataFrame, assets: list[str], n: int = 20) -> pd.DataFrame:
    rows = []
    for _, e in ev.iterrows():
        p0 = int(e["t0_pos"]) - 1
        if p0 - n < 0 or p0 + n >= len(rets):
            continue
        pre, post = rets.iloc[p0 - n:p0][assets], rets.iloc[p0:p0 + n][assets]
        rows.append(pd.DataFrame({"pre": pre.std() * np.sqrt(252), "post": post.std() * np.sqrt(252)}).assign(event=e["event"]))
    v = pd.concat(rows).rename_axis("asset").reset_index()
    out = v.groupby("asset")[["pre", "post"]].median()
    out["change"] = out["post"] / out["pre"] - 1
    out["p_wilcoxon"] = v.groupby("asset").apply(lambda g: stats.wilcoxon(g["post"], g["pre"]).pvalue if len(g) >= 5 else np.nan)
    return out.sort_values("change", ascending=False)

ASSETS = list(res.ar.columns)
vol_table = vol_shift(rets, EV, ASSETS)
vol_table.style.format({"pre": "{:.1%}", "post": "{:.1%}", "change": "{:+.1%}", "p_wilcoxon": "{:.3f}"})

## 10 · Visualisations
A consistent, print-friendly style. Each chart answers one question and is saved as a 200-dpi PNG to `outputs/figures/` for the README.

In [ ]:
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.alpha": 0.25, "grid.linestyle": "-", "legend.frameon": False,
})
PALETTE = {"BZ=F": "#1f4e79", "CL=F": "#2e75b6", "NG=F": "#c55a11", "BDRY": "#7f6000",
           "FRO": "#548235", "BTU": "#3a3a3a", "XLE": "#7030a0", "GC=F": "#bf9000"}
from matplotlib.ticker import PercentFormatter, MaxNLocator
def pct(ax, axis="y"):
    """Percent ticks on 'nice' steps so labels never round misleadingly (e.g. 7.5% → 8%)."""
    a = getattr(ax, f"{axis}axis")
    a.set_major_locator(MaxNLocator(nbins=7, steps=[1, 2, 2.5, 5, 10]))
    a.set_major_formatter(PercentFormatter(1.0, decimals=None))

def save(fig, name):
    path = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(path)
    print("saved →", path)

### 10.1 · Average CAR path around the event, with 95% bootstrap bands

In [ ]:
def car_paths(ar: pd.DataFrame, t_start: int) -> pd.DataFrame:
    """Cumulate ARs from t_start so every path starts at 0 just before the window."""
    return ar.groupby(level="event", sort=False).cumsum()

paths = car_paths(res.ar, -CFG.pre)
focus = [a for a in ["BZ=F", "NG=F", "FRO", "BDRY", "BTU", "GC=F"] if a in ASSETS]

fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True)
for ax, a in zip(axes.flat, focus):
    wide = paths[a].unstack("tau")                     # events × tau
    mean = wide.mean()
    boots = np.array([wide.sample(len(wide), replace=True, random_state=i).mean() for i in range(1000)])
    lo, hi = np.nanquantile(boots, [0.025, 0.975], axis=0)
    ax.fill_between(mean.index, lo, hi, color=PALETTE[a], alpha=0.15, lw=0)
    ax.plot(mean.index, mean, color=PALETTE[a], lw=2.2)
    ax.axvline(0, color="crimson", lw=1, ls="--")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(CFG.assets[a], fontsize=11)
    ax.set_xticks(range(-CFG.pre, CFG.post + 1, 5))
    pct(ax)
for ax in axes[-1]:
    ax.set_xlabel("Trading days relative to event (τ)")
fig.suptitle(f"Cumulative abnormal returns around {len(res.meta)} geopolitical energy shocks",
             x=0.01, y=0.995, ha="left", fontsize=14, fontweight="bold")
fig.text(0.01, 0.945, "Mean CAR (line) and 95% bootstrap band; dashed line = event day", fontsize=9, color="dimgray")
fig.tight_layout(rect=(0, 0, 1, 0.92))
save(fig, "01_car_paths"); plt.show()

### 10.2 · Which assets react most? Mean CAR[0,+5] with confidence intervals

In [ ]:
s = summary[key].sort_values("mean_CAR")
fig, ax = plt.subplots(figsize=(9, 5))
colors = [PALETTE.get(a, "gray") for a in s.index]
bars = ax.barh(s["name"], s["mean_CAR"], color=colors)
for b, sig in zip(bars, s["significant"]):
    b.set_alpha(1.0 if sig else 0.4)
ax.errorbar(s["mean_CAR"], s["name"], xerr=[s["mean_CAR"] - s["ci_low"], s["ci_high"] - s["mean_CAR"]],
            fmt="none", ecolor="black", elinewidth=1, capsize=3)
span = (s["ci_high"].max() - s["ci_low"].min())
for y, (m, p, lo, hi) in enumerate(zip(s["mean_CAR"], s["p_cs"], s["ci_low"], s["ci_high"])):
    stars = "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""
    # label sits just beyond the end of the CI whisker, so it never overlaps the bar or error bar
    xpos, ha = (hi + span * 0.015, "left") if m >= 0 else (lo - span * 0.015, "right")
    ax.text(xpos, y, f"{m:+.1%}{stars}", va="center", ha=ha, fontsize=9, fontweight="bold")
ax.set_xlim(s["ci_low"].min() - span * 0.15, s["ci_high"].max() + span * 0.15)
ax.axvline(0, color="black", lw=0.8)
pct(ax, "x")
ax.set_title(f"Mean CAR over [0, +5] days after the event")
ax.set_xlabel("Cumulative abnormal return   (faded = not significant at 5%;  * p<.10  ** p<.05  *** p<.01)")
ax.grid(axis="y", visible=False)
save(fig, "02_mean_car_bars"); plt.show()

### 10.3 · Event × asset heatmap: which shocks moved what?

In [ ]:
C = car(res.ar, 0, 5)
C.index = [f"{res.meta.loc[e, 't0']:%Y-%m-%d}  {e}" for e in C.index]
C = C.rename(columns=CFG.assets)

lim = np.nanpercentile(np.abs(C.to_numpy()), 95)
fig, ax = plt.subplots(figsize=(11, 0.42 * len(C) + 1.5))
im = ax.imshow(C.to_numpy(), cmap="RdBu_r", norm=TwoSlopeNorm(0, -lim, lim), aspect="auto")
ax.set_xticks(range(C.shape[1]), C.columns, rotation=35, ha="right")
ax.set_yticks(range(C.shape[0]), C.index, fontsize=8.5)
for i in range(C.shape[0]):
    for j in range(C.shape[1]):
        v = C.iat[i, j]
        if pd.notna(v):
            ax.text(j, i, f"{v:+.0%}", ha="center", va="center", fontsize=7.5,
                    color="white" if abs(v) > lim * 0.6 else "black")
ax.grid(False)
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01)
cb.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:+.0%}"))
ax.set_title("CAR[0,+5] by event and asset (red = abnormal gain, blue = abnormal loss)")
save(fig, "03_event_heatmap"); plt.show()

### 10.4 · Prices around the shocks: Brent with events marked

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(px_clean.index, px_clean["BZ=F"], color=PALETTE["BZ=F"], lw=1.2)
cat_colors = {"chokepoint": "#c00000", "conflict": "#7030a0", "supply": "#c55a11"}
for _, e in res.meta.iterrows():
    ax.axvline(e["t0"], color=cat_colors[e["category"]], alpha=0.55, lw=1)
for c, col in cat_colors.items():
    ax.plot([], [], color=col, label=c)
ax.legend(title="Event type", loc="upper left")
ax.set_ylabel("USD / barrel")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_title("Brent crude with geopolitical events marked")
save(fig, "04_brent_timeline"); plt.show()

### 10.5 · Volatility before vs after

In [ ]:
v = vol_table.rename(index=CFG.assets).sort_values("post")
fig, ax = plt.subplots(figsize=(9, 4.8))
y = np.arange(len(v))
ax.hlines(y, v["pre"], v["post"], color="lightgray", lw=3)
ax.scatter(v["pre"], y, color="gray", s=45, label="20 days before", zorder=3)
ax.scatter(v["post"], y, color="crimson", s=45, label="20 days after", zorder=3)
ax.set_yticks(y, v.index)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.set_xlabel("Annualised realised volatility (median across events)")
ax.set_title("Realised volatility before vs after geopolitical shocks")
ax.legend(loc="lower right"); ax.grid(axis="y", visible=False)
save(fig, "05_vol_shift"); plt.show()

## 11 · Performance metrics and a simple trading lens
An event study measures *reaction*, not a tradable strategy, because events can't be known in advance. Still, a practical question is: **if you bought the reaction at the close on the event day (τ=0) and held for 5 days, did the move continue or reverse?** That tests whether markets fully price the shock on day 0 (efficient) or under-react (drift).

A positive CAR[+1,+5] means continuation (under-reaction); a negative one means reversal (over-reaction).

In [ ]:
def drift_metrics(res: EventResult) -> pd.DataFrame:
    d0 = res.ar.xs(0, level="tau")
    follow = car(res.ar, 1, 5)
    rows = []
    for a in d0.columns:
        x0, x1 = d0[a].dropna(), follow[a].dropna()
        both = pd.concat([x0, x1], axis=1, join="inner")
        same_dir = (np.sign(both.iloc[:, 0]) == np.sign(both.iloc[:, 1])).mean()
        hit = (x1 > 0).mean()
        sharpe = x1.mean() / x1.std() * np.sqrt(len(x1)) if x1.std() > 0 else np.nan
        rows.append(dict(asset=a, name=CFG.assets[a], day0_AR=x0.mean(), CAR_1_5=x1.mean(),
                         continuation_rate=same_dir, hit_rate=hit, t_stat=sharpe,
                         worst=x1.min(), best=x1.max()))
    return pd.DataFrame(rows).set_index("asset")

drift = drift_metrics(res)
drift.style.format({"day0_AR": "{:+.2%}", "CAR_1_5": "{:+.2%}", "continuation_rate": "{:.0%}",
                    "hit_rate": "{:.0%}", "t_stat": "{:.2f}", "worst": "{:+.1%}", "best": "{:+.1%}"})

## 12 · Export results

In [ ]:
with pd.ExcelWriter(os.path.join(OUT_DIR, "event_study_results.xlsx")) as xw:
    for k, df in summary.items():
        df.to_excel(xw, sheet_name=("CAR " + k.strip("[]").replace(",", " to "))[:31])
    car(res.ar, 0, 5).to_excel(xw, sheet_name="CAR by event")
    cat_table.to_excel(xw, sheet_name="By category")
    vol_table.to_excel(xw, sheet_name="Volatility")
    drift.to_excel(xw, sheet_name="Drift")
    res.meta.to_excel(xw, sheet_name="Events")
res.ar.to_csv(os.path.join(OUT_DIR, "abnormal_returns.csv"))
print("Exported:", sorted(os.listdir(OUT_DIR)))

## 13 · Auto-generated findings

In [ ]:
def findings(summary, drift, vol_table, cat_table) -> str:
    s = summary["[0,+5]"]
    top = s.iloc[0]
    sig = s[s["significant"]]
    lines = [
        f"Across {int(s['n'].max())} geopolitical energy shocks since {CFG.start[:4]}:",
        f"• Largest average reaction: {top['name']} at {top['mean_CAR']:+.1%} over 5 days "
        f"(95% CI {top['ci_low']:+.1%} to {top['ci_high']:+.1%}; positive in {top['pct_positive']:.0%} of events).",
        f"• Statistically significant at 5%: {', '.join(sig['name']) if len(sig) else 'none, so treat the results as directional'}.",
    ]
    if "GC=F" in s.index:
        lines.append(f"• Gold, the safe-haven control, moved {s.loc['GC=F','mean_CAR']:+.1%}.")
    vmax = vol_table["change"].idxmax()
    lines.append(f"• Volatility rose most for {CFG.assets[vmax]} (+{vol_table.loc[vmax,'change']:.0%} median).")
    if "chokepoint" in cat_table and "BDRY" in cat_table.index:
        lines.append(f"• Chokepoint events moved dry-bulk freight {cat_table.loc['BDRY','chokepoint']:+.1%} on average.")
    cont = drift["continuation_rate"].idxmax()
    lines.append(f"• Strongest post-event drift: {CFG.assets[cont]} continued in the day-0 direction "
                 f"{drift.loc[cont,'continuation_rate']:.0%} of the time.")
    lines.append("Caveats: small sample of events, overlapping windows in 2019 and 2025, and the event dates are hand-picked.")
    return "\n".join(lines)

print(findings(summary, drift, vol_table, cat_table))

## 14 · Limitations and next steps
* **Small N.** With about 16 events, power is limited. Bootstrap CIs and the sign test help, but don't over-claim.
* **Clustering.** The June 2019 and June 2025 events overlap. Clustered standard errors (Kolari & Pynnönen, 2010) would be the rigorous fix.
* **Event selection bias.** Events chosen in hindsight are the ones that "mattered". A GDELT news-based event feed would remove that.
* **Futures roll.** Yahoo continuous futures (`BZ=F`) splice contracts. Using front-month roll-adjusted series from the EIA would be cleaner.
* **Coal proxy.** There's no liquid free coal futures feed, so Peabody equity stands in for coal prices, with added equity noise.

**Upgrades:** GARCH-based abnormal volatility · GDELT-driven event detection · India-specific assets (Coal India, INR) · an interactive Streamlit dashboard.